**Last updated: 2026-11-25** - Two-port quarter-wave Purcell filter based on [Jeffrey et al., Fig. 1](https://arxiv.org/pdf/1401.0257).

2026-11-23 - Substrate set to MIT LL BISQuit8: 725 um silicon, eps_r 11.45, as its own HFSS material (was HFSS library silicon, 11.9, at Metal's default 750 um); the HFSS calibrations, measured on 11.9, are rescaled by (eps_r + 1) / 2. LP_W1 moved to the 20-pin package's southern west pad (y = -1.8 mm, MIT LL pad via mitll_pad.py), Purcell short to -1.95 mm, qubit column (claws) to x = -0.1 mm, as in firstpass.ipynb. Readout arm pocket 40 um shorter (175.45 um) and claw 40 um shorter (inner 160 um), matching the per-arm qubits in firstpass.ipynb.

2026-11-22: Turned 90 degrees with the rest of the chip: the qubits are a vertical chain, the claws on their west arms, and the feedline runs south to north down the west side, read out from LP_W1.

2026-11-17: - Feedline, resonators and claws only (no qubits, junctions or EPR). Purcell tail too short to draw is now added to the short stub (fixes HFSS PK_ERROR_crossing_edge). HFSS runs in AEDT 2024 R2.

# Readout: Purcell-filter feedline and readout resonators

**To change the design:** edit the numbers in the *Design parameters* cell, then **Restart & Run All**.

The five quarter-wave readout resonators and the shared feedline. W4 drives the filter through a series interdigital capacitor at the voltage antinode. W1 receives the transmitted signal through a tap near the grounded end. The target filter center is 7.5 GHz and the target power bandwidth is 800 MHz.

Each resonator ends in its claw, drawn where it is on the chip but without the qubit (`readout_claw.py`; ground plane where the qubit would be). The qubits, couplers and control lines are in `qubits_couplers.ipynb`, and the whole chip (with the GDS export) is in `firstpass.ipynb`.

In [ ]:
import importlib
import os

import numpy as np

os.environ.setdefault("QISKIT_METAL_HEADLESS", "True")

from qiskit_metal import designs, Dict
import qiskit_metal as qm
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee
from qiskit_metal.qlibrary.couplers.line_tee import LineTee
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.lumped.cap_n_interdigital import CapNInterdigital
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.anchored_path import RouteAnchors

# Local file next to this notebook. A running kernel can miss a file added
# after it started, so refresh the import system's view of the folder first.
importlib.invalidate_caches()
from readout_claw import ReadoutClaw  # local file: readout_claw.py
from mitll_pad import MitllLaunchpad, load_package_pads, pad_placement  # local file: mitll_pad.py

print("Notebook version: readout.ipynb, last updated 2026-11-25")

## Design parameters

In [ ]:
# =====================================================================
#  DESIGN PARAMETERS
#  Every number you are likely to tune lives in this cell. Edit, then
#  Restart & Run All. The suffix on each name is its unit (_um / _mm).
# =====================================================================

# --- Chip ------------------------------------------------------------
chip_size_mm = 5.0  # MIT LL standard die (SQP3.1 DRM 2.9)
# Substrate: MIT LL BISQuit8 (DRM 2.1.1), 725 um high-resistivity silicon,
# eps_r 11.45 at low temperature. HFSS gets a material of its own for it:
# its library "silicon" is eps_r 11.9 (and Metal's default chip 750 um thick).
substrate_thickness_um = 725
substrate_eps_r = 11.45
substrate_material = "silicon_bisquit"
# The HFSS calibrations below were measured on HFSS's library silicon. A
# planar layout's capacitances all scale with (eps_r + 1) / 2, so they are
# rescaled to this substrate by hfss_eps_scale; set hfss_calibration_eps_r =
# substrate_eps_r once they are re-measured on it.
hfss_calibration_eps_r = 11.9
hfss_eps_scale = (substrate_eps_r + 1) / (hfss_calibration_eps_r + 1)
cpw_width_um = 10  # default centre trace for every route
cpw_gap_um = 6

# --- Readout claws -------------------------------------------------------
# Each resonator ends in a claw around a qubit's west arm. The qubits are
# not drawn here (see readout_claw.py), but the claws sit where they are on
# the chip, so these numbers are the qubits' (from qubits_couplers.ipynb;
# copy them across if they change there). The qubits are a vertical chain at
# orientation 90, Q0 at the south end.
n_resonators = 5
qubit_pitch_mm = 0.75
qubit_column_x_mm = -0.1
cross_etch_side_arm_length_um = 175.45  # qubit pocket of the readout (west) arm: centre square edge to tip
cross_etch_arm_width_um = 89.1  # qubit pocket: width of each arm

# Readout claw on the west arm: a U of metal wrapped around the arm tip.
# Sizes are of the claw metal itself. "Length" runs along the arm,
# "width" across it. The ground spacing (claw gap to cross gap) follows
# from these and the cross sizes above.
claw_outer_length_um = 170.9  # shortened with the readout arm
claw_outer_width_um = 128.9
claw_inner_length_um = 160
claw_inner_width_um = 107.1
claw_gap_um = 6

# --- Readout --------------------------------------------------------------
# Target bare quarter-wave frequencies, ReadoutRes0..4. The meander lengths are
# computed from these (see the resonator cell), so edit frequencies, not lengths.
resonator_freqs_ghz = [7.3, 7.4, 7.5, 7.6, 7.7]
# HFSS calibration of the resonator lengths. With the lengths computed for
# 7.3-7.7 GHz from (eps_r + 1) / 2 alone, HFSS (full_chip, 2026-11-10) gave
# these frequencies, about 10 % low: the claw and tee capacitances load each
# resonator, and the real effective permittivity differs a little. 1/f_HFSS
# is a straight line in 1/f_design (to 1.3 MHz here), and the resonator
# cell inverts that line to choose each length. Lengths are computed for
# 7.3 -> 6.607 etc., not frequencies the chip had. Replace with new HFSS
# numbers and the lengths they were computed for, to refine.
# Measured on library silicon, so rescaled to the substrate (see Chip).
resonator_hfss_calibration_ghz = {
    k: v / hfss_eps_scale**0.5
    for k, v in {7.3: 6.607, 7.4: 6.684, 7.5: 6.763, 7.6: 6.843, 7.7: 6.921}.items()
}
resonator_meander_spacing_um = 50
resonator_lead_um = 150  # straight section at each end of the meander
resonator_fillet_um = 20

# Two-port filter: W4 -> series capacitor -> quarter-wave CPW -> short.
# W1 is the output tap. Its placement and the input capacitance both load the mode.
purcell_filter_freq_ghz = 7.5
purcell_filter_bw_ghz = 0.8
purcell_filter_q = purcell_filter_freq_ghz / purcell_filter_bw_ghz
purcell_z0_ohm = 50  # first-pass CPW and port impedances
# Effective phase velocity seed from the previous loaded filter's HFSS run.
purcell_filter_hfss_factor = 7.10 / 7.5 / hfss_eps_scale**0.5
feedline_x_mm = -1.5
purcell_short_y_mm = -1.68
purcell_tap_y_mm = -1.04  # between CLT0 and CLT1
# IDC starting geometry. Extract the mutual capacitance in Q3D/HFSS and tune
# the fingers to purcell_input_cap_ff, reported by the circuit model below.
purcell_idc_fingers = 11
purcell_idc_finger_length_um = 60
purcell_idc_finger_width_um = 4
purcell_idc_gap_um = 2
purcell_idc_lead_um = 30
tee_coupling_length_um = 200
tee_coupling_space_um = 4

# --- Launchpads W4 (input) and W1 (output) ------------------------------------
# The northern and southern west-edge pads of MIT LL's 20-pin package (mitll_pad.py), as in
# firstpass.ipynb: 160 x 200 um, its back 200 um in from the edge, with a
# 100 um taper to the CPW.
package_gds = "HIERARCHY_EXAMPLE_v3_1_1.gds"
package_cell = "TEST04"  # the 20-pin package
pad_taper_um = 100

# --- Routing --------------------------------------------------------
feedline_fillet_um = 50

# --- HFSS simulations -------------------------------------------------
# Run in Ansys HFSS through Metal's "hfss" renderer, which starts AEDT if it
# is not already open. Each takes from minutes to an hour or more.
# eigenmode, W4 and W1 as 50 ohm resistors: frequency and Q of every readout and
# of the filter itself, so each readout's kappa
run_hfss_purcell_eigenmode = True
run_hfss_purcell = True  # driven modal: S21 from W4 to W1
hfss_purcell_sweep_ghz = (6.5, 8.5, 4001)  # start, stop, points (0.5 MHz steps)
hfss_max_passes = 20
hfss_basis_order = -1  # 0 zero, 1 first, 2 second, -1 mixed
hfss_max_delta_f = 0.1  # eigenmode convergence, % change in frequency per pass
hfss_min_converged = 2  # passes in a row that must meet it before HFSS stops
hfss_max_delta_s = 0.02  # driven-modal convergence, change in S per pass
hfss_buffer_mm = 0.3  # substrate and vacuum beyond what is rendered
hfss_project_file = "readout_hfss.aedt"  # new each run; _2, _3... if taken
hfss_results_dir = "hfss_results/readout"  # mode tables, as CSV

## Readout claws

In [ ]:
design = designs.DesignPlanar()
design.chips.main.size.size_x = f"{chip_size_mm}mm"
design.chips.main.size.size_y = f"{chip_size_mm}mm"
design.chips.main.size.size_z = f"-{substrate_thickness_um}um"
design.chips.main.material = substrate_material  # made in AEDT by the HFSS cell
design.variables.cpw_width = f"{cpw_width_um}um"
design.variables.cpw_gap = f"{cpw_gap_um}um"
design.overwrite_enabled = True

pocket_tip_um = cross_etch_arm_width_um / 2 + cross_etch_side_arm_length_um  # centre to etched arm tip

# Claw metal thicknesses and the ground spacing, from the outer/inner sizes.
claw_side_um = (claw_outer_width_um - claw_inner_width_um) / 2
claw_back_um = claw_outer_length_um - claw_inner_length_um
claw_ground_spacing_um = (claw_inner_width_um - cross_etch_arm_width_um) / 2 - claw_gap_um
if claw_ground_spacing_um < 0:
    print(
        f"WARNING: claw inner width {claw_inner_width_um} um is too narrow for the "
        f"{cross_etch_arm_width_um} um arm pocket plus the claw gap. Ground spacing = "
        f"{claw_ground_spacing_um:.2f} um: the claw overlaps the pocket."
    )
# Distance from qubit centre to the claw finger tips, vs. the E/W arm pocket.
claw_tip_um = pocket_tip_um + claw_ground_spacing_um + claw_gap_um - claw_inner_length_um
if claw_tip_um - claw_gap_um < cross_etch_arm_width_um / 2:
    print(
        f"WARNING: claw fingers reach {claw_tip_um:.1f} um from the qubit centre and "
        f"would run into the qubit's E/W arm pocket (edge at {cross_etch_arm_width_um / 2} um)."
    )


def qubit_y(i):
    """y position (mm) of qubit i (the centre its claw wraps around)."""
    return (i - (n_resonators - 1) / 2) * qubit_pitch_mm


claws = [
    ReadoutClaw(
        design,
        f"Claw{i}",
        options=dict(
            pos_x=f"{qubit_column_x_mm}mm",
            pos_y=f"{qubit_y(i)}mm",
            orientation="90",
            pocket_width=f"{cross_etch_arm_width_um}um",
            pocket_tip=f"{pocket_tip_um}um",
            connection_pads=dict(
                readout=dict(
                    connector_type="0",
                    claw_length=f"{claw_inner_length_um}um",
                    claw_width=f"{claw_side_um}um",
                    claw_width_back=f"{claw_back_um}um",
                    ground_spacing=f"{claw_ground_spacing_um}um",
                    claw_gap=f"{claw_gap_um}um",
                    connector_location="90",
                )
            ),
        ),
    )
    for i in range(n_resonators)
]

print(f"Design initialized on a {chip_size_mm} x {chip_size_mm} mm chip.")
print(f"Placed readout claws Claw0-Claw{n_resonators - 1}.")

## Readout and feedline

In [ ]:
# Each tee is shifted so its coupled-line end sits directly west of the
# claw, keeping every resonator a straight horizontal meander. Tees on
# the south half are mirrored (coupled end on their south side), which keeps
# the column of tees symmetric about the chip centre.
readout_tees = []
for i in range(n_resonators):
    mirror = i < (n_resonators - 1) / 2
    shift_mm = (1 if mirror else -1) * tee_coupling_length_um / 2e3
    readout_tees.append(
        CoupledLineTee(
            design,
            f"CLT{i}",
            options=dict(
                pos_x=f"{feedline_x_mm}mm",
                pos_y=f"{qubit_y(i) + shift_mm}mm",
                orientation="90",
                coupling_length=f"{tee_coupling_length_um}um",
                coupling_space=f"{tee_coupling_space_um}um",
                mirror=mirror,
                # Short the resonator end to ground: with the open claw at
                # the qubit, each readout resonator is a quarter-wave.
                open_termination=False,
            ),
        )
    )

west_pads = load_package_pads(package_gds, package_cell)["W"]
for pad_name, (along_um, back_um) in (("LP_W1", west_pads[0]), ("LP_W4", west_pads[-1])):
    MitllLaunchpad(
        design, pad_name,
        options=pad_placement("W", along_um, back_um, chip_size_mm,
                              taper_height=f"{pad_taper_um}um"),
    )

print("Placed readout tees and launchpads W4 (input) and W1 (output).")

In [ ]:
from scipy.optimize import brentq, least_squares, minimize_scalar

c_mm_ghz = 299.792458
eps_eff = (substrate_eps_r + 1) / 2
purcell_velocity = c_mm_ghz / eps_eff**0.5 * purcell_filter_hfss_factor
tap_from_short_mm = purcell_tap_y_mm - purcell_short_y_mm


def purcell_power(freq_ghz, length_mm, cap_ff):
    # ABCD cascade normalized to Z0: series C, CPW to the output tap,
    # then the shorted CPW stub in shunt with the matched output port.
    beta = 2 * np.pi * freq_ghz / purcell_velocity
    phase = beta * (length_mm - tap_from_short_mm)
    cap_z = 1 / (2j * np.pi * freq_ghz * 1e9 * cap_ff * 1e-15 * purcell_z0_ohm)
    stub_y = -1j / np.tan(beta * tap_from_short_mm)
    c, js = np.cos(phase), 1j * np.sin(phase)
    a, b = c + cap_z * js, js + cap_z * c
    return abs(2 / (a + js + (b + c) * (1 + stub_y)))**2


def passband(frequency, power):
    # Interpolate the nearest crossings on either side of the broad peak.
    peak = np.argmax(power)
    half = power[peak] / 2
    below = np.flatnonzero(power[:peak] <= half)
    above = np.flatnonzero(power[peak + 1:] <= half) + peak + 1
    if not below.size or not above.size:
        raise ValueError("The frequency sweep does not span both half-power crossings.")
    lo, hi = below[-1], above[0]
    low = np.interp(half, power[lo:lo + 2], frequency[lo:lo + 2])
    high = np.interp(half, power[hi - 1:hi + 1][::-1], frequency[hi - 1:hi + 1][::-1])
    return low, high


def model_band(length_mm, cap_ff):
    f0 = purcell_filter_freq_ghz
    peak = minimize_scalar(lambda f: -purcell_power(f, length_mm, cap_ff),
                           bounds=(0.7 * f0, 1.3 * f0), method="bounded")
    def half_power(f):
        return purcell_power(f, length_mm, cap_ff) + peak.fun / 2
    lo = brentq(half_power, 0.6 * f0, peak.x)
    hi = brentq(half_power, peak.x, 1.5 * f0)
    return peak.x, lo, hi


def band_error(values):
    peak, lo, hi = model_band(*values)
    return [peak - purcell_filter_freq_ghz, hi - lo - purcell_filter_bw_ghz]


quarter_wave_mm = purcell_velocity / (4 * purcell_filter_freq_ghz)
fit = least_squares(band_error, (0.93 * quarter_wave_mm, 50),
                    bounds=((0.75 * quarter_wave_mm, 1), (1.05 * quarter_wave_mm, 150)))
if np.linalg.norm(fit.fun) > 1e-4:
    raise ValueError("Retune the output tap position for this center and bandwidth.")
purcell_target_mm, purcell_input_cap_ff = fit.x


def pin_xy(component, pin):
    return design.components[component].pins[pin].middle


def connect(name, start, end, anchors=()):
    options = dict(pin_inputs=dict(start_pin=dict(zip(("component", "pin"), start)),
                                   end_pin=dict(zip(("component", "pin"), end))))
    if anchors:
        options.update(anchors=Dict(enumerate(anchors)), fillet=f"{feedline_fillet_um}um",
                       advanced=Dict(avoid_collision=False))
        return RouteAnchors(design, name, options=options)
    return RouteStraight(design, name, options=options)


tap_half_mm = 0.025
LineTee(design, "Purcell_tap", options=dict(pos_x=f"{feedline_x_mm}mm",
        pos_y=f"{purcell_tap_y_mm}mm", orientation="270", t_length=f"{tap_half_mm}mm"))
ShortToGround(design, "Purcell_short", options=dict(pos_x=f"{feedline_x_mm}mm",
              pos_y=f"{purcell_short_y_mm}mm", orientation="270"))
connect("Purcell_stub", ("Purcell_short", "short"), ("CLT0", "prime_start"))
connect("Feed_mid_0_south", ("CLT0", "prime_end"), ("Purcell_tap", "prime_end"))
connect("Feed_mid_0_north", ("Purcell_tap", "prime_start"), ("CLT1", "prime_start"))
for i in range(1, n_resonators - 1):
    connect(f"Feed_mid_{i}", (f"CLT{i}", "prime_end"), (f"CLT{i + 1}", "prime_start"))

pad_x, output_y = pin_xy("LP_W1", "tie")
output_bend_x = (pad_x + feedline_x_mm) / 2
connect("Feed_out", ("LP_W1", "tie"), ("Purcell_tap", "second_end"),
        ((output_bend_x, output_y), (output_bend_x, purcell_tap_y_mm)))

# Put the filter-side IDC reference plane at the center of the fingers.
# The north end of the backbone bends west into the W4 input.
last_tee = (f"CLT{n_resonators - 1}", "prime_end")
last_y = pin_xy(*last_tee)[1]
input_x, pad_input_y = pin_xy("LP_W4", "tie")
input_y = last_y + 0.065
cap_half_mm = (purcell_idc_gap_um + 2 * purcell_idc_finger_width_um
               + purcell_idc_finger_length_um) / 2e3
cap_pin_to_center_mm = cap_half_mm + purcell_idc_lead_um / 1e3
bend_trim_mm = (2 - np.pi / 2) * feedline_fillet_um / 1e3
cap_x = feedline_x_mm - (purcell_target_mm - (input_y - purcell_short_y_mm) + bend_trim_mm)
if not input_x < cap_x - cap_pin_to_center_mm < cap_x + cap_pin_to_center_mm < feedline_x_mm:
    raise ValueError("The input capacitor does not fit between the W4 pad and the filter backbone.")
cap = CapNInterdigital(design, "Purcell_input_cap", options=dict(
    pos_x=f"{cap_x - cap_pin_to_center_mm}mm", pos_y=f"{input_y}mm", orientation="90",
    cap_width=f"{purcell_idc_finger_width_um}um", cap_gap=f"{purcell_idc_gap_um}um",
    cap_gap_ground="10um", finger_length=f"{purcell_idc_finger_length_um}um",
    finger_count=purcell_idc_fingers, cap_distance=f"{purcell_idc_lead_um}um",
))
tail = connect("Purcell_tail", last_tee, ("Purcell_input_cap", "south_end"),
               ((feedline_x_mm, input_y),))
input_bend_x = (input_x + cap_x - cap_pin_to_center_mm) / 2
connect("Feed_in", ("LP_W4", "tie"), ("Purcell_input_cap", "north_end"),
        ((input_bend_x, pad_input_y), (input_bend_x, input_y)))
purcell_length_mm = last_y - purcell_short_y_mm + tail.length + cap_pin_to_center_mm
peak, lo, hi = model_band(purcell_length_mm, purcell_input_cap_ff)
print(f"Filter CPW {purcell_length_mm:.3f} mm, output tap {tap_from_short_mm:.3f} mm from ground")
print(f"Target input capacitance: {purcell_input_cap_ff:.1f} fF (IDC extraction pending)")
print(f"Circuit model: peak {peak:.3f} GHz, -3 dB band {lo:.3f}-{hi:.3f} GHz "
      f"({(hi - lo) * 1e3:.0f} MHz)")

In [ ]:
# Quarter-wave length from frequency, with the CPW effective permittivity
# (eps_r + 1) / 2. The resonator also includes the claw's CPW stub and the
# tee's coupled section, so those are taken off the meander's length. The
# claw and tee capacitances pull the real frequency down, so each length is
# computed for a higher design frequency, from the HFSS calibration in the
# parameters: 1/f_HFSS = a / f_design + b. (c_mm_ghz and eps_eff are defined
# in the feedline cell above.)
cal_design = np.array(list(resonator_hfss_calibration_ghz))
cal_hfss = np.array(list(resonator_hfss_calibration_ghz.values()))
cal_a, cal_b = np.polyfit(1 / cal_design, 1 / cal_hfss, 1)
resonator_design_freqs_ghz = [cal_a / (1 / f - cal_b) for f in resonator_freqs_ghz]
claw_stub_mm = claws[0].p.connection_pads.readout.claw_cpw_length
tee_section_mm = readout_tees[0].p.coupling_length + readout_tees[0].p.down_length
resonator_lengths_mm = [
    c_mm_ghz / (4 * f * eps_eff**0.5) - claw_stub_mm - tee_section_mm
    for f in resonator_design_freqs_ghz
]

for i in range(n_resonators):
    RouteMeander(
        design,
        f"ReadoutRes{i}",
        options=dict(
            pin_inputs=dict(
                start_pin=dict(component=f"Claw{i}", pin="readout"),
                end_pin=dict(component=f"CLT{i}", pin="second_end"),
            ),
            total_length=f"{resonator_lengths_mm[i]}mm",
            fillet=f"{resonator_fillet_um}um",
            lead=dict(
                start_straight=f"{resonator_lead_um}um",
                end_straight=f"{resonator_lead_um}um",
            ),
            meander=dict(spacing=f"{resonator_meander_spacing_um}um", asymmetry="0um"),
        ),
    )

print("Staggered meander resonators routed.")
for i, (f, fd, length) in enumerate(
    zip(resonator_freqs_ghz, resonator_design_freqs_ghz, resonator_lengths_mm)
):
    print(f"  ReadoutRes{i}: {f:.3f} GHz in HFSS ({fd:.3f} from eps_eff alone) -> meander {length:.3f} mm")
span_mhz = (max(resonator_freqs_ghz) - min(resonator_freqs_ghz)) * 1e3
print(f"  band {min(resonator_freqs_ghz):.2f}-{max(resonator_freqs_ghz):.2f} GHz, span {span_mhz:.0f} MHz")

In [ ]:
fig = qm.view(design)
fig.suptitle("Readout layout")
fig

## HFSS simulations

The parameters cell controls the two HFSS workflows (`run_hfss_*`).

* **Purcell eigenmode.** Both ports terminated in 50 ohms. Reports mode frequencies and Q, including the readout linewidths.
* **Purcell readout (driven modal).** Measures transmission from W4 to W1. A filter-only run measures the broad -3 dB bandwidth, and a second run includes all five resonators.

There are no qubits here, so the qubit-side numbers (dispersive shift, Purcell-limited qubit T1) need the full chip in `firstpass.ipynb`.

Each run of these cells starts a new AEDT project, saved next to this notebook as `hfss_project_file`, or with `_2`, `_3`, ... added if that file exists.

In [ ]:
# Shared by the HFSS cells below: a new AEDT project for this run.
hfss_on = run_hfss_purcell_eigenmode or run_hfss_purcell
hfss_results = {}

if hfss_on:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from qiskit_metal.analyses.simulation import EigenmodeSim, ScatteringImpedanceSim

    # A new AEDT project for this run, saved at once under an unused name.
    # The renderer then attaches to it by name before every simulation, so
    # the designs cannot land in whichever project happens to be active.
    stem, ext = os.path.splitext(os.path.abspath(hfss_project_file))
    hfss_project_path, k = stem + ext, 2
    while os.path.exists(hfss_project_path):
        hfss_project_path, k = f"{stem}_{k}{ext}", k + 1
    hfss = design.renderers.hfss
    # AEDT 2024 R2. Started only if no 2024 R2 copy is open already: the
    # renderer attaches to an AEDT that is already running, so starting
    # another (or a different version) just leaves extra windows, and several
    # open copies can stall the connection.
    import subprocess

    running = subprocess.run(
        ["powershell", "-NoProfile", "-Command",
         "(Get-Process ansysedt -ErrorAction SilentlyContinue).Path"],
        capture_output=True, text=True,
    ).stdout
    if "v242" not in running:
        design.renderers.hfss.open_ansys(path_var="ANSYSEM_ROOT242")
    hfss.start()
    hfss.rdesktop.new_project().save(hfss_project_path)
    hfss.options.project_name = os.path.splitext(os.path.basename(hfss_project_path))[0]
    hfss.options.project_path = None
    hfss.start()  # reconnect, now to the new project
    print(f"HFSS project: {hfss_project_path} (AEDT {hfss.rdesktop.get_version()})")
    # The substrate (see Chip in the parameters), as a project material that
    # every design in the project draws the chip from.
    hfss.pinfo.project._project.GetDefinitionManager().AddMaterial(
        [f"NAME:{substrate_material}", "CoordinateSystemType:=", "Cartesian",
         "BulkOrSurfaceType:=", 1, ["NAME:PhysicsTypes", "set:=", ["Electromagnetic"]],
         "permittivity:=", str(substrate_eps_r), "dielectric_loss_tangent:=", "0"]
    )
    print(f"Substrate: {substrate_material}, eps_r {substrate_eps_r}, {substrate_thickness_um} um")



def hfss_buffer(sim):
    sim.renderer.options.x_buffer_width_mm = hfss_buffer_mm
    sim.renderer.options.y_buffer_width_mm = hfss_buffer_mm


# The parts every simulation renders: the whole feedline, the resonators and
# their claws.
feedline_parts = ["Feed_in", "Feed_out", "Purcell_input_cap", "Purcell_tap",
                  "Purcell_stub", "Purcell_short", "Purcell_tail",
                  "Feed_mid_0_south", "Feed_mid_0_north"]
feedline_parts += [f"CLT{i}" for i in range(n_resonators)]
feedline_parts += [f"Feed_mid_{i}" for i in range(1, n_resonators - 1)]
purcell_ports = [("Feed_in", "start", 50), ("Feed_out", "start", 50)]
readout_parts = [part for i in range(n_resonators) for part in (f"ReadoutRes{i}", f"Claw{i}")]

In [ ]:
# Purcell eigenmode: the feedline and every readout, with W4 and W1 terminated in 50 ohms.
# Each mode's Q reflects leakage into both ports:
#   - the filter: the lowest-Q mode (Q ~ purcell_filter_q)
#   - each readout: kappa / 2 pi = f / Q
# The modes are named by frequency and Q alone.
if run_hfss_purcell_eigenmode:
    pe = EigenmodeSim(design, "hfss")
    hfss_buffer(pe)
    pe.setup.update(
        n_modes=n_resonators + 2,  # readouts, filter, one spare
        min_freq_ghz=0.8 * min(resonator_freqs_ghz),
        max_passes=hfss_max_passes,
        min_converged=hfss_min_converged,
        max_delta_f=hfss_max_delta_f,
        basis_order=hfss_basis_order,
    )
    pe.run(
        name="Purcell_eigenmode",
        components=feedline_parts + readout_parts,
        open_terminations=[],
        port_list=purcell_ports,
    )
    pe.plot_convergences()

    freqs, kappas = pe.renderer.pinfo.setup.get_solutions().eigenmodes()
    f_ghz = np.asarray(freqs, dtype=float)
    k_ghz = np.asarray(kappas if kappas is not None else [np.nan] * len(f_ghz), dtype=float)
    if f_ghz.max() > 1e3:  # exported in Hz
        f_ghz, k_ghz = f_ghz / 1e9, k_ghz / 1e9
    q = f_ghz / k_ghz

    # Names: the lowest-Q mode is the filter; the rest are the readouts in
    # frequency order, then anything extra.
    order = list(np.argsort(f_ghz))
    filt = min(order, key=lambda m: q[m])
    labels = {filt: "PurcellFilter"}
    others = [m for m in order if m != filt]
    for i, m in enumerate(others):
        labels[m] = f"ReadoutRes{i}" if i < n_resonators else f"extra{i - n_resonators}"

    targets = {f"ReadoutRes{i}": f for i, f in enumerate(resonator_freqs_ghz)}
    targets["PurcellFilter"] = purcell_filter_freq_ghz
    purcell_modes = pd.DataFrame(
        {
            labels[m]: {
                "f (GHz)": f_ghz[m],
                "target (GHz)": targets.get(labels[m], np.nan),
                "Q": q[m],
                "kappa/2pi (MHz)": k_ghz[m] * 1e3,
            }
            for m in order
        }
    ).T
    os.makedirs(hfss_results_dir, exist_ok=True)
    purcell_modes.to_csv(os.path.join(hfss_results_dir, "Purcell_eigenmode_modes.csv"))
    hfss_results["Purcell_eigenmode"] = purcell_modes
    print("Purcell eigenmode, W4 and W1 terminated in 50 ohms:")
    print(purcell_modes.to_string(float_format=lambda v: f"{v:.4g}"))

In [ ]:
# Driven transmission of the filter alone, then with all five readouts.
if run_hfss_purcell:
    drv = ScatteringImpedanceSim(design, "hfss")
    hfss_buffer(drv)
    start_ghz, stop_ghz, count = hfss_purcell_sweep_ghz
    drv.setup.update(
        freq_ghz=purcell_filter_freq_ghz,
        max_passes=hfss_max_passes,
        max_delta_s=hfss_max_delta_s,
        min_converged=hfss_min_converged,
        basis_order=hfss_basis_order,
    )
    drv.setup.sweep_setup.update(
        start_ghz=start_ghz, stop_ghz=stop_ghz, count=count, step_ghz=None, type="Fast"
    )
    os.makedirs(hfss_results_dir, exist_ok=True)
    for label, parts in (("Purcell_filter_only", feedline_parts),
                         ("Purcell_readout", feedline_parts + readout_parts)):
        drv.run(name=label, components=parts, open_terminations=[], port_list=purcell_ports)
        transmission = "S21"
        _, _, s_params = drv.renderer.get_params([transmission])
        f_ghz = np.asarray(s_params.index, dtype=float)
        s21 = np.asarray(s_params[transmission], dtype=complex)
        power = abs(s21)**2
        if label == "Purcell_filter_only":
            lo, hi = passband(f_ghz, power)
            print(f"HFSS filter -3 dB bandwidth: {(hi - lo) * 1e3:.0f} MHz "
                  f"({lo:.3f}-{hi:.3f} GHz)")
        fig, ax = plt.subplots()
        ax.plot(f_ghz, 10 * np.log10(np.maximum(power, np.finfo(float).tiny)))
        for freq in resonator_freqs_ghz:
            ax.axvline(freq, color="grey", linestyle=":", alpha=0.5)
        ax.set(xlabel="Frequency (GHz)", ylabel="|S21| (dB)", title=label)
        s_params.to_csv(os.path.join(hfss_results_dir, f"{label}_s21.csv"))
        hfss_results[label] = dict(f_ghz=f_ghz, s21=s21)

In [ ]:
# Save the AEDT project (every simulation above is a design in it) and
# release the connection to AEDT. AEDT itself stays open for inspection.
if hfss_on:
    hfss.pinfo.project.save()
    print(f"Saved the HFSS project to {hfss_project_path}")
    hfss.stop()